In [1]:
# ==== MNI audit + T1w/MNI viewer (auto-detect output dirs) ====
from pathlib import Path
import re, numpy as np, nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output
from functools import lru_cache
from scipy.ndimage import binary_dilation



# BASE (the directory that directly contains mni_1mm_ants/ and/or mni_1mm_ants_fixed/)
ROOT = Path("/home/rbielski/Atlas_2/Registered")
TRAIN_IMAGES = Path("/home/rbielski/Atlas_2/Training/Images")
CANDIDATE_MNI_DIRS = [ROOT / "mni_1mm_ants_fixed", ROOT / "mni_1mm_ants"]
OUT_MNI = next((p for p in CANDIDATE_MNI_DIRS if p.exists()), None)
print("ROOT:", ROOT)
print("Found OUT_MNI:", OUT_MNI)


OUT_NATMSK = ROOT / "native_resampled_masks"  # optional
HAVE_NATIVE = OUT_NATMSK.exists()

TPL = Path("/home/rbielski/.cache/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-01_desc-brain_T1w.nii.gz")
tpl_img = nib.load(str(TPL))
tpl_shape = tpl_img.shape[:3]
tpl_zooms = tuple(float(z) for z in tpl_img.header.get_zooms()[:3])
tpl_aff   = tpl_img.affine

def _key_from_name(p: Path) -> str:
    m_sub = re.search(r"(sub-[^_]+)", p.name)
    m_ses = re.search(r"(ses-[^_]+)", p.name)
    parts = [m_sub.group(1) if m_sub else None, m_ses.group(1) if m_ses else None]
    return "_".join([x for x in parts if x])

@lru_cache(maxsize=256)
def _load_img(path: str) -> nib.Nifti1Image: return nib.load(path)

@lru_cache(maxsize=256)
def _load_vol(path: str) -> np.ndarray:
    arr = nib.load(path).get_fdata()
    if arr.ndim == 4 and arr.shape[-1] == 1: arr = arr[..., 0]
    return arr.astype(np.float32)

def _normalize(img: np.ndarray) -> np.ndarray:
    nz = img[img > 0]
    if nz.size == 0: return np.zeros_like(img, dtype=np.float32)
    p1, p99 = np.percentile(nz, [1, 99]); img = np.clip(img, p1, p99)
    m, s = nz.mean(), nz.std(); img = (img - m) / (s + 1e-8)
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn + 1e-8)

def _edges2d(m2d): 
    from scipy.ndimage import binary_dilation
    m = m2d.astype(bool); return binary_dilation(m) & (~m)

def _zooms3(img: nib.Nifti1Image):
    z = img.header.get_zooms()[:3]
    return tuple(float(v) for v in z)

def _affine_equal(a: np.ndarray, b: np.ndarray, tol=1e-4): 
    return np.allclose(a, b, atol=tol)

# ---- Collect MNI pairs ----
t1_mni = sorted(OUT_MNI.glob("*_T1w_MNI.nii.gz"))
mask_mni_by_base = {p.name.replace("_lesion_mask_MNI.nii.gz",""): p
                    for p in OUT_MNI.glob("*_lesion_mask_MNI.nii.gz")}
PAIRS = {}
for t1p in t1_mni:
    key  = _key_from_name(t1p)
    base = t1p.name.replace("_T1w_MNI.nii.gz", "")
    mskp = mask_mni_by_base.get(base)
    if mskp: PAIRS[key] = {"t1_mni": t1p, "mask_mni": mskp}

assert PAIRS, f"No *_T1w_MNI / *_lesion_mask_MNI pairs found in {OUT_MNI}"

# ---- MNI audit ----
bad = []
for k, v in PAIRS.items():
    ti = _load_img(str(v["t1_mni"])); mi = _load_img(str(v["mask_mni"]))
    ok_shape = (ti.shape[:3] == tpl_shape) and (mi.shape[:3] == tpl_shape)
    ok_zooms = (_zooms3(ti) == tpl_zooms) and (_zooms3(mi) == tpl_zooms)
    ok_aff_t1  = _affine_equal(ti.affine, tpl_aff)
    ok_aff_msk = _affine_equal(mi.affine, tpl_aff)
    if not (ok_shape and ok_zooms and ok_aff_t1 and ok_aff_msk):
        bad.append(dict(
            key=k, t1=v["t1_mni"].name, msk=v["mask_mni"].name,
            t1_shape=ti.shape[:3], msk_shape=mi.shape[:3],
            t1_zooms=_zooms3(ti), msk_zooms=_zooms3(mi),
            t1_aff_ok=ok_aff_t1, msk_aff_ok=ok_aff_msk
        ))

print(f"[MNI audit] dir={OUT_MNI.name} | total pairs: {len(PAIRS)} | OK: {len(PAIRS)-len(bad)} | FAIL: {len(bad)}")
if bad:
    for row in bad[:10]: print(row)

# ---- Optional native view wiring (if you created resampled masks) ----
if HAVE_NATIVE:
    for k in list(PAIRS.keys()):
        t1_native = next(iter(TRAIN_IMAGES.glob(f"{k}*T1w.nii.gz")), None)
        m_native  = OUT_NATMSK / f"{k}_lesion_mask_T1w_native.nii.gz"
        if t1_native and m_native.exists():
            PAIRS[k]["t1_native"] = t1_native
            PAIRS[k]["mask_t1"]   = m_native

# ---- Viewer ----
keys_sorted = sorted(PAIRS.keys())
pair_dd   = W.Dropdown(options=keys_sorted, description="Case:", layout=W.Layout(width="100%"))
slice_sl  = W.IntSlider(description="Axial slice:", min=0, max=1, value=0, continuous_update=False, layout=W.Layout(width="60%"))
alpha_sl  = W.FloatSlider(description="Mask α:", min=0.0, max=1.0, step=0.05, value=0.55, layout=W.Layout(width="35%"))
edges_cb  = W.Checkbox(description="Edges only", value=True)
invert_cb = W.Checkbox(description="Invert image", value=False)

def _bg_options_for(key: str):
    opts = [("T1w_MNI (final)", "MNI")]
    if "t1_native" in PAIRS[key] and "mask_t1" in PAIRS[key]:
        opts.append(("T1w_native (with native mask)", "NATIVE"))
    return opts

bg_radio = W.RadioButtons(options=_bg_options_for(keys_sorted[0]), value="MNI",
                          description="Background:", layout=W.Layout(width="40%"))

status = W.HTML(f"<b>Viewer</b> — cases: {len(keys_sorted)}")
controls = W.VBox([status, pair_dd, W.HBox([slice_sl, alpha_sl]), W.HBox([edges_cb, invert_cb, bg_radio])])
out = W.Output()

def _update_bg_options(*_):
    key = pair_dd.value
    bg_radio.options = _bg_options_for(key)
    if bg_radio.value not in [v for _, v in bg_radio.options]:
        bg_radio.value = bg_radio.options[0][1]

def _update_slice_range(*_):
    key = pair_dd.value
    if bg_radio.value == "MNI":
        vol = _load_vol(str(PAIRS[key]["t1_mni"]))
    else:
        vol = _load_vol(str(PAIRS[key]["t1_native"]))
    slice_sl.max = max(0, int(vol.shape[2] - 1))
    slice_sl.value = min(slice_sl.value, slice_sl.max)

def _draw(*_):
    with out:
        clear_output(wait=True)
        try:
            key = pair_dd.value
            if bg_radio.value == "MNI":
                img_p  = PAIRS[key]["t1_mni"];  mask_p = PAIRS[key]["mask_mni"];  bg_name = "T1w_MNI"
                ti = _load_img(str(img_p)); mi = _load_img(str(mask_p))
                aff_ok = _affine_equal(ti.affine, tpl_aff) and _affine_equal(mi.affine, tpl_aff)
                shp_ok = (ti.shape[:3] == tpl_shape) and (mi.shape[:3] == tpl_shape)
                z_ok   = (_zooms3(ti) == tpl_zooms) and (_zooms3(mi) == tpl_zooms)
                audit = f" | MNI check: shape {'✅' if shp_ok else '⚠️'}, zooms {'✅' if z_ok else '⚠️'}, affine {'✅' if aff_ok else '⚠️'}"
            else:
                img_p  = PAIRS[key]["t1_native"]; mask_p = PAIRS[key]["mask_t1"];  bg_name = "T1w_native"; audit = ""

            img_hdr = _load_img(str(img_p)); img = _load_vol(str(img_p))
            msk_hdr = _load_img(str(mask_p)); msk = (_load_vol(str(mask_p)) > 0.5)

            z_img = _zooms3(img_hdr); z_msk = _zooms3(msk_hdr)
            aff_eq = _affine_equal(img_hdr.affine, msk_hdr.affine)
            shp_eq = img_hdr.shape[:3] == msk_hdr.shape[:3]
            status.value = (f"<b>Viewer</b> — cases: {len(keys_sorted)} | {bg_name}: "
                            f"shape {img_hdr.shape[:3]} vs mask {msk_hdr.shape[:3]} "
                            f"| zooms {tuple(round(v,3) for v in z_img)} / {tuple(round(v,3) for v in z_msk)} "
                            f"| affines match: {'✅' if aff_eq else '⚠️'} | shapes match: {'✅' if shp_eq else '⚠️'}{audit}")

            img_view = _normalize(img.copy())
            if invert_cb.value: img_view = 1.0 - img_view
            idx = int(slice_sl.value)
            img2d = img_view[:, :, idx]; m2d = msk[:, :, idx]

            plt.figure(figsize=(5.6, 5.6))
            plt.imshow(img2d.T, cmap="gray", origin="lower")
            if edges_cb.value:
                plt.contour(_edges2d(m2d).T, levels=[0.5], linewidths=0.8, colors="r")
            else:
                plt.imshow(np.ma.masked_where(~m2d.T, m2d.T), cmap="jet", alpha=float(alpha_sl.value), origin="lower")
            plt.axis("off"); plt.tight_layout(); plt.show(); plt.close()

            print(f"Image: {img_p.name}\nMask : {mask_p.name}\nDir: {OUT_MNI}")

        except Exception as exc:
            print("Draw error:", exc)

def _refresh(*_):
    _update_bg_options(); _update_slice_range(); _draw()

pair_dd.observe(_refresh, names="value")
slice_sl.observe(_draw, names="value")
alpha_sl.observe(_draw, names="value")
edges_cb.observe(_draw, names="value")
invert_cb.observe(_draw, names="value")
bg_radio.observe(_refresh, names="value")

_refresh()
display(controls, out)


ROOT: /home/rbielski/Atlas_2/Registered
Found OUT_MNI: /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed
[MNI audit] dir=mni_1mm_ants_fixed | total pairs: 16 | OK: 16 | FAIL: 0


Output()

In [2]:
from pathlib import Path
import pandas as pd, numpy as np

OUT = Path("/home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed")
qc = pd.read_csv(OUT / "qc_summary.csv")

# mask volume (mm^3) from header zooms * voxel count
def parse_tuple(s):
    # strings like "(193, 229, 193)" or "(1.0, 1.0, 1.0)"
    s = s.strip().strip("()")
    return tuple(float(x) for x in s.split(","))
vol_mm3 = []
for z_str, nvox in zip(qc["mask_zooms"], qc["mask_nonzero"]):
    z = parse_tuple(z_str)
    vol_mm3.append(nvox * (z[0]*z[1]*z[2]))
qc["mask_vol_mm3"] = vol_mm3

print("Pairs:", len(qc))
print("Mask voxel counts — min/median/max:", int(qc["mask_nonzero"].min()),
      int(qc["mask_nonzero"].median()), int(qc["mask_nonzero"].max()))
print("Mask volume (ml) — min/median/max:",
      round(qc["mask_vol_mm3"].min()/1000,2),
      round(qc["mask_vol_mm3"].median()/1000,2),
      round(qc["mask_vol_mm3"].max()/1000,2))

# flag anything suspiciously tiny/huge
sus = qc[(qc["mask_vol_mm3"] < 1_000) | (qc["mask_vol_mm3"] > 200_000)]  # <1 ml or >200 ml
print("\nSuspicious volumes:", len(sus))
print(sus[["key","mask_nonzero","mask_vol_mm3"]].head(10).to_string(index=False))


FileNotFoundError: [Errno 2] No such file or directory: '/home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/qc_summary.csv'

In [ ]:
# === ATLAS T1w + lesion-masks: metrics + rich summaries + sample rows (native & MNI) ===
from pathlib import Path
import re
import numpy as np
import pandas as pd
import nibabel as nib
from math import prod
from IPython.display import display

# ---- Directories ----
ROOT    = Path("/home/rbielski/Atlas_2/Registered")
TRAIN_IMAGES = Path("/home/rbielski/Atlas_2/Training/Images")
OUT_NAT = ROOT / "native_resampled_masks"   # *_lesion_mask_T1w_native.nii.gz
OUT_MNI = ROOT / "mni_1mm_ants_fixed"       # *_T1w_MNI.nii.gz + *_lesion_mask_MNI.nii.gz
assert OUT_NAT.exists(), f"Missing {OUT_NAT}"
assert OUT_MNI.exists(), f"Missing {OUT_MNI}"

# ---- Helpers ----
def _tag(s, t):
    m = re.search(fr"({t}-[^_]+)", s)
    return m.group(1) if m else None

def key_from_name(p: Path) -> str:
    sub = _tag(p.name, "sub")
    ses = _tag(p.name, "ses")
    return "_".join([x for x in (sub, ses) if x])

def t1_pref(name: str) -> int:
    n = name.lower()
    if "tfl" in n: return 0
    if "mprage" in n: return 1
    if "mp2rage" in n: return 2
    return 3

def choose_native_t1(key: str) -> Path | None:
    cands = sorted(TRAIN_IMAGES.glob(f"{key}*T1w.nii.gz"))
    if not cands: return None
    cands.sort(key=lambda p: (t1_pref(p.name), p.name))
    return cands[0]

def img_metrics(img_path: Path, nonzero_thresh=0.0):
    img = nib.load(str(img_path))
    shp = img.shape[:3]
    z   = tuple(float(v) for v in img.header.get_zooms()[:3])
    data = img.get_fdata()
    nz  = int(np.count_nonzero(data > nonzero_thresh))
    vox_total = int(prod(shp))
    fov_mm = (shp[0]*z[0], shp[1]*z[1], shp[2]*z[2])
    voxel_vol_mm3 = z[0]*z[1]*z[2]
    fov_vol_ml = (vox_total * voxel_vol_mm3) / 1000.0
    return dict(
        shape_x=shp[0], shape_y=shp[1], shape_z=shp[2],
        zooms_x=z[0], zooms_y=z[1], zooms_z=z[2],
        vox_total=vox_total,
        img_nonzero=nz,
        brain_frac=(nz/vox_total if vox_total else np.nan),
        fov_mm_x=fov_mm[0], fov_mm_y=fov_mm[1], fov_mm_z=fov_mm[2],
        voxel_vol_mm3=voxel_vol_mm3,
        fov_vol_ml=fov_vol_ml,
    )

def mask_metrics(mask_path: Path, zooms_xyz: tuple[float,float,float]):
    m = nib.load(str(mask_path))
    vox = int(np.count_nonzero(m.get_fdata() > 0.5))
    ml  = (vox * zooms_xyz[0] * zooms_xyz[1] * zooms_xyz[2]) / 1000.0
    return dict(mask_voxels=vox, mask_ml=ml)

# ---- Build rows driven by MNI outputs ----
t1_mni_files = sorted(OUT_MNI.glob("*_T1w_MNI.nii.gz"))
assert t1_mni_files, f"No *_T1w_MNI.nii.gz found in {OUT_MNI}"

rows = []
for t1_mni in t1_mni_files:
    key = key_from_name(t1_mni)
    msk_mni = OUT_MNI / f"{key}_lesion_mask_MNI.nii.gz"
    if not msk_mni.exists():
        continue

    # Native counterparts
    t1_nat = choose_native_t1(key)
    msk_nat = OUT_NAT / f"{key}_lesion_mask_T1w_native.nii.gz"

    # MNI metrics
    mni_img = img_metrics(t1_mni, nonzero_thresh=0.0)
    mni_mask = mask_metrics(msk_mni, (mni_img["zooms_x"], mni_img["zooms_y"], mni_img["zooms_z"]))

    # Native metrics
    if t1_nat and msk_nat.exists():
        nat_img = img_metrics(t1_nat, nonzero_thresh=0.0)
        nat_mask = mask_metrics(msk_nat, (nat_img["zooms_x"], nat_img["zooms_y"], nat_img["zooms_z"]))
    else:
        nat_img = {k: np.nan for k in [
            "shape_x","shape_y","shape_z","zooms_x","zooms_y","zooms_z","vox_total",
            "img_nonzero","brain_frac","fov_mm_x","fov_mm_y","fov_mm_z","voxel_vol_mm3","fov_vol_ml"
        ]}
        nat_mask = {"mask_voxels": np.nan, "mask_ml": np.nan}
        t1_nat = None
        msk_nat = None

    rows.append({
        "key": key,
        # Native
        "nat_shape_x": nat_img["shape_x"], "nat_shape_y": nat_img["shape_y"], "nat_shape_z": nat_img["shape_z"],
        "nat_zooms_x": nat_img["zooms_x"], "nat_zooms_y": nat_img["zooms_y"], "nat_zooms_z": nat_img["zooms_z"],
        "nat_vox_total": nat_img["vox_total"], "nat_img_nonzero": nat_img["img_nonzero"], "nat_brain_frac": nat_img["brain_frac"],
        "nat_fov_mm_x": nat_img["fov_mm_x"], "nat_fov_mm_y": nat_img["fov_mm_y"], "nat_fov_mm_z": nat_img["fov_mm_z"],
        "nat_voxel_vol_mm3": nat_img["voxel_vol_mm3"], "nat_fov_vol_ml": nat_img["fov_vol_ml"],
        "nat_mask_voxels": nat_mask["mask_voxels"], "nat_mask_ml": nat_mask["mask_ml"],
        # MNI
        "mni_shape_x": mni_img["shape_x"], "mni_shape_y": mni_img["shape_y"], "mni_shape_z": mni_img["shape_z"],
        "mni_zooms_x": mni_img["zooms_x"], "mni_zooms_y": mni_img["zooms_y"], "mni_zooms_z": mni_img["zooms_z"],
        "mni_vox_total": mni_img["vox_total"], "mni_img_nonzero": mni_img["img_nonzero"], "mni_brain_frac": mni_img["brain_frac"],
        "mni_fov_mm_x": mni_img["fov_mm_x"], "mni_fov_mm_y": mni_img["fov_mm_y"], "mni_fov_mm_z": mni_img["fov_mm_z"],
        "mni_voxel_vol_mm3": mni_img["voxel_vol_mm3"], "mni_fov_vol_ml": mni_img["fov_vol_ml"],
        "mni_mask_voxels": mni_mask["mask_voxels"], "mni_mask_ml": mni_mask["mask_ml"],
    })

df = pd.DataFrame(rows).sort_values("key").reset_index(drop=True)

# Save CSV
csv_path = OUT_MNI / "image_mask_metrics.csv"
df.to_csv(csv_path, index=False)
print(f"Wrote: {csv_path}")
print("Pairs:", len(df))

# ===== Summaries =====
def summary_block(prefix: str, label: str):
    ok = df[f"{prefix}_img_nonzero"].notna()
    if not ok.any():
        print(f"\n=== {label} ===\n(no data)")
        return
    vox   = df.loc[ok, f"{prefix}_img_nonzero"].to_numpy(int)
    total = df.loc[ok, f"{prefix}_vox_total"].to_numpy(int)
    frac  = df.loc[ok, f"{prefix}_brain_frac"].to_numpy(float)
    fovml = df.loc[ok, f"{prefix}_fov_vol_ml"].to_numpy(float)
    mvox  = df.loc[ok, f"{prefix}_mask_voxels"].to_numpy(int)
    mml   = df.loc[ok, f"{prefix}_mask_ml"].to_numpy(float)

    pct = lambda a, q: np.percentile(a, q)
    print(f"\n=== {label} ===")
    print("ICV proxy (nonzero voxels):")
    print(f"  min/median/max: {vox.min():,} / {int(pct(vox,50)):,} / {vox.max():,}")
    print("Brain fraction of FOV (nonzero/total):")
    print(f"  mean±sd: {frac.mean():.3f} ± {frac.std():.3f} | p10/50/90: {pct(frac,10):.3f} / {pct(frac,50):.3f} / {pct(frac,90):.3f}")
    print("FOV volume (mL):")
    print(f"  min/median/max: {fovml.min():.1f} / {pct(fovml,50):.1f} / {fovml.max():.1f}")
    print("Lesion mask volume (mL):")
    print(f"  min/median/max: {mml.min():.2f} / {pct(mml,50):.2f} / {mml.max():.2f}")
    print("Lesion mask voxels:")
    print(f"  min/median/max: {mvox.min():,} / {int(pct(mvox,50)):,} / {mvox.max():,}")

summary_block("nat", "NATIVE summary")
summary_block("mni", "MNI summary")

# ===== Sample tables (first 10 rows) =====
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

native_cols = [
    "key",
    "nat_shape_x","nat_shape_y","nat_shape_z",
    "nat_zooms_x","nat_zooms_y","nat_zooms_z",
    "nat_vox_total","nat_img_nonzero","nat_brain_frac",
    "nat_fov_mm_x","nat_fov_mm_y","nat_fov_mm_z","nat_fov_vol_ml",
    "nat_mask_voxels","nat_mask_ml",
]
mni_cols = [
    "key",
    "mni_shape_x","mni_shape_y","mni_shape_z",
    "mni_zooms_x","mni_zooms_y","mni_zooms_z",
    "mni_vox_total","mni_img_nonzero","mni_brain_frac",
    "mni_fov_mm_x","mni_fov_mm_y","mni_fov_mm_z","mni_fov_vol_ml",
    "mni_mask_voxels","mni_mask_ml",
]
combined_cols = [
    "key",
    "nat_shape_x","nat_shape_y","nat_shape_z","nat_zooms_x","nat_zooms_y","nat_zooms_z",
    "nat_vox_total","nat_img_nonzero","nat_mask_voxels","nat_mask_ml",
    "mni_shape_x","mni_shape_y","mni_shape_z","mni_zooms_x","mni_zooms_y","mni_zooms_z",
    "mni_vox_total","mni_img_nonzero","mni_mask_voxels","mni_mask_ml"
]

print("\n=== NATIVE (first 10 rows) ===")
display(df[native_cols].head(10))
print("\n=== MNI (first 10 rows) ===")
display(df[mni_cols].head(10))
print("\n=== COMBINED (first 10 rows) ===")
display(df[combined_cols].head(10))
